In [ ]:
from proto import *
from engine import *
from utils import *
from runners import CompartmentalModel
from experimental import *

import pandas as pd
import numpy as np
import datetime as dt

pd.options.plotting.backend = "plotly"

In [ ]:
import computegraph as cg

In [ ]:
class Parameter(cg.Variable):
    def __init__(self, key, default):
        super().__init__(key, "parameters")
        self.default = default


class ModelVariable(cg.Variable):
    def __init__(self, key):
        super().__init__(key, "model_variables")
        self.key = key


Time = ModelVariable("time")
CompartmentValues = ModelVariable("compartment_values")
param = Parameter


def defer(func, name=None):
    def _proxy(*args, **kwargs):
        return cg.Function(func, args, kwargs, name)

    return _proxy


def label(graph_obj, name):
    graph_obj.node_name = name
    return graph_obj

In [ ]:
from utils import LinearInterpolator

In [ ]:
from computegraph.types import Data

In [ ]:
epoch = Epoch(dt.datetime(1980, 6, 7))

In [ ]:
x = jnp.array([0.0, 15.0, 20.0, 40.0])
y = jnp.array([0.0, 0.0, 2.0, 0.0])

t = LinearInterpolator(x, y)

t.process(jnp.linspace(0.0, 50.0))

In [ ]:
double_time = defer(lambda t: t * 2.0, "doubletime")(Time * 2.0)
vinterp = defer(LinearInterpolator, "CRInterpolator")(
    Data(x, "cr_times"), Data(y, "cr_values")
)

In [ ]:
vacct = defer(LinearInterpolator.process)(vinterp, Time)

In [ ]:
disease_state = Stratification("disease_state", ["S", "I", "R"])
humans = CompartmentMap.new(disease_state)
humans.compartments

In [ ]:
age_strat = humans.stratify(Stratification("age", ["child", "adult"]))

In [ ]:
loc_strat = humans.stratify(Stratification("location", ["N", "S", "E", "W"]))

In [ ]:
class InfectionProcess:
    def __init__(
        self,
        mm: ManagedArray,
        infectee_cats: CategoryGroup,
        infector_cats: CategoryGroup,
    ):
        self.mm = mm
        self.infector_cats = infector_cats
        self.infectee_cats = infectee_cats

    def process(self, compartment_values: ManagedArray, contact_rate: float):
        ipops = compartment_values.sumcats("compartment", self.infector_cats)
        total_pop = compartment_values.query(compartment=disease_state[...]).data.sum()
        age_foi = self.mm.data @ ipops.data / total_pop * contact_rate
        return CategoryData(self.infectee_cats, age_foi)

In [ ]:
disease_state["S", "I"]

In [ ]:
age_strat.categories()

In [ ]:
mmnp = np.array([[1.8, 0.3], [0.2, 0.4]])

spatial_mm = np.random.normal(1.0, 0.1, (4, 4))

jnp.kron(mmnp, spatial_mm)

mm = ManagedArray(mmnp, ["dest", "source"])
mm.indices["source"] = ManagedCategoryGroupIndex("source", age_strat.categories())
mm.indices["dest"] = ManagedCategoryGroupIndex("dest", age_strat.categories())

infectees = age_strat.categories().product(disease_state["S"])
infectors = age_strat.categories().product(disease_state["I"])

iprocess = defer(InfectionProcess)(mm, infectees, infectors)

In [ ]:
class ManagedMatrix(ManagedArray):
    def __init__(self, data, source: CategoryGroup, dest: CategoryGroup):
        indices = {
            "source": ManagedCategoryGroupIndex("source", source),
            "dest": ManagedCategoryGroupIndex("dest", dest),
        }
        super().__init__(data=data, dims=["source", "dest"], indices=indices)

In [ ]:
smm = ManagedMatrix(spatial_mm, loc_strat.categories(), loc_strat.categories())
amm = ManagedMatrix(mmnp, age_strat.categories(), age_strat.categories())

In [ ]:
jnp.kron(smm.data, amm.data)

In [ ]:
amm.data * smm.data[0, 0]

In [ ]:
c0 = humans.compartments[0]
c1 = humans.compartments[1]

In [ ]:
StratSpec

In [ ]:
Category(age_strat["child", "adult"]).traits[0][1]

In [ ]:
Category([age_strat["child", "adult"]]).matches(
    Category([age_strat["child", "adult"]]), age_strat
)

In [ ]:
amm.indices["source"].index.product(smm.indices["source"].index)

In [ ]:
class CategoryContainer:
    compartments: CompartmentArray

    def __init__(
        self,
        compartments: CompartmentArray,
        root: "CompartmentMap" = None,
        parent: "CompartmentContainer" = None,
        indices: np.array = None,
    ):
        self.compartments = np.array(compartments)
        if parent is None and indices is None:
            parent = self
            indices = np.arange(len(compartments))

        if parent is not None and indices is not None:
            self.parent = parent
            self.indices = indices
        else:
            raise Exception("Both or neither of parent and indices must be specified")

        self.root = root

    def __getitem__(self, indices):
        compartments = self.compartments[indices]
        return CompartmentContainer(compartments, self.root, self, indices)

    def query(self, traits: list[StratSpec]) -> "CompartmentContainer":
        traits = validate_qspec(traits)
        qres = []
        indices = []
        for i, c in enumerate(self.compartments):
            has_all = True
            for t in traits:
                has_trait = False
                for sspec in iter_stratspec(t):
                    has_trait = has_trait or sspec in c.strata
                if not has_trait:
                    has_all = False
                    break
            if has_all:
                qres.append(c)
                indices.append(i)
        return CompartmentContainer(np.array(qres), self.root, self, np.array(indices))

    def wrap_data(self, data):
        return CompartmentDataContainer(self.compartments, self, data)

    def zeros(self, lib=jnp):
        return CompartmentDataContainer(
            self.compartments, self, lib.zeros(len(self.compartments))
        )

    def __repr__(self):
        if self.parent == self:
            return "CompartmentContainer:\n" + repr(self.compartments)
        else:
            return (
                f"CompartmentContainer view of 0x{id(self.parent)}:\n"
                + repr(self.compartments)
                + repr(self.indices)
            )

    def __len__(self):
        return len(self.compartments)

    def get_labels(self):
        def compname(c: Compartment):
            return "_".join([stratum for (strat, stratum) in c.strata])

        return [compname(c) for c in self.compartments]

In [ ]:
cc = CompartmentContainer(
    np.array(
        [
            Compartment(prop)
            for prop in amm.indices["source"]
            .index.product(smm.indices["source"].index)
            .categories
        ]
    )
)

In [ ]:
humans.compartments[0]

In [ ]:
humans.compartments[0].matches(humans.compartments[1], [disease_state])

In [ ]:
jnp.kron(amm.data, smm.data)

In [ ]:
jnp.kron(smm.data, amm.data)

In [ ]:
smm = ManagedArray(spatial_mm, ["dest", "source"])
mm.indices["source"] = ManagedCategoryGroupIndex("source", loc_strat.categories())
mm.indices["dest"] = ManagedCategoryGroupIndex("source", loc_strat.categories())

In [ ]:
mm.data

In [ ]:
cvals = humans.zeros(np)
cvals.data[:] = 1.0

In [ ]:
type(infectors)

In [ ]:
get_cat_indices(infectors, humans)

In [ ]:
cvma = ManagedArray(
    cvals.data,
    dims=["compartment"],
    indices={"compartment": ManagedIndex("compartment", cvals)},
)

In [ ]:
cvma.query(compartment=age_strat["child"])

In [ ]:
ca = ManagedArray(
    cvals.data,
    dims=["compartment"],
    indices={"compartment": ManagedIndex("compartment", cvals)},
).sumcats("compartment", infectors)

In [ ]:
cvals

In [ ]:
infectors

In [ ]:
ca

In [ ]:
ca["category"].query?

In [ ]:
ManagedArray(
    cvals.data,
    dims=["compartment"],
    indices={"compartment": ManagedIndex("compartment", cvals)},
).sumcats("compartment", infectors).data

In [ ]:
infectors

In [ ]:
foi = defer(InfectionProcess.process)(
    iprocess, CompartmentValues, Parameter("contact_rate", 0.2)
)

In [ ]:
infection = TransitionFlow(disease_state["S"], disease_state["I"], foi)
recovery = TransitionFlow(
    disease_state["I"], disease_state["R"], Parameter("recovery_rate", 0.1)
)

In [ ]:
flows = {"infection": infection, "recovery": recovery}
model = CompartmentalModel(humans, flows)

runner = model.get_runner(50, epoch)

runner.graph.draw()

In [ ]:
istate = humans.zeros(np).data
S_idx = humans.query(disease_state["S"]).indices
istate[S_idx] = 100.0
istate[humans.query(age_strat["child"]).indices] *= 0.5
istate[humans.query([loc_strat["N"], disease_state["I"]]).indices] = 1.0
istate[humans.query([loc_strat["W"], disease_state["I"]]).indices] = 2.0
params = {"contact_rate": 2.1, "recovery_rate": 0.1}

results = runner.run(istate, params)

In [ ]:
compres = results["compartments"]
flowres = results["flows"]

In [ ]:
def compname(c: Compartment):
    return "_".join([stratum for (strat, stratum) in c.strata])


comp_labels = [compname(c) for c in model.cmap.compartments]

pd.DataFrame(
    index=compres.indices["time"].index, data=compres.data, columns=comp_labels
).plot()

In [ ]:
infdata = flowres["infection"]

In [ ]:
infdata.to_pandas_df().plot()

In [ ]:
infdata.sumcats("source", loc_strat.categories()).to_pandas_df().plot()

In [ ]:
loc_strat.categories()

In [ ]:
compres.to_pandas_df().plot()

In [ ]:
flo

In [ ]:
compres.query(compartment=disease_state["I"]).sumcats(
    "compartment", loc_strat.categories()
).to_pandas_df().plot()

In [ ]:
comp_results["compartments"].query(compartment=disease_state["I"]).sumcats(
    "compartment", loc_strat.categories()
)

In [ ]:
all_child_inf = comp_results["flows"]["infection"].query(source=age_strat["child"])

In [ ]:
infdata = comp_results["flows"]["infection"]

In [ ]:
age_inf_res = comp_results["flows"]["infection"].sum(
    cats=("source", age_strat.categories())
)

In [ ]:
age_inf_res.indices["time"]

In [ ]:
pd.DataFrame(
    age_inf_res.data,
    index=age_inf_res.indices["time"].index,
    columns=age_inf_res.indices["category"].index,
)

In [ ]:
infdata.indices["source"].index.get_labels()

In [ ]:
comp_results["flows"]["infection"].sum(cats=("source", age_strat.categories())).query(
    category=["child"]
).data.shape

In [ ]:
qbackref = pd.Series(
    index=comp_results["flows"]["infection"]
    .sum(cats=("source", age_strat.categories()))
    .indices["category"][1],
    data=np.arange(2),
)[["child"]]

In [ ]:
category_idx_reduction

In [ ]:
infdata = comp_results["flows"]["infection"]

In [ ]:
def flow(c: Compartment):
    return "_".join([stratum for (strat, stratum) in c.strata])


comp_labels = [compname(c) for c in model.cmap.compartments]

In [ ]:
age_cats = age_strat.categories()

In [ ]:
infdata.data.shape

In [ ]:
cidx = cat_indices(age_cats, humans)
jnp.array([infdata.data[:, c].sum(axis=-1) for c in cidx]).shape

In [ ]:
infdata.data[:, np.array(cidx)].sum(axis=-1).shape

In [ ]:
from experimental import LA

In [ ]:
def category_names(cat_groups):
    return [
        "_".join(["|".join([stratum for stratum in strata]) for strat, strata in cat])
        for cat in cat_groups
    ]

In [ ]:
def masum(ma: ManagedArray, cats=None, dims=None):
    if cats is not None:
        idx_name, catgroups = cats
        maps_dim, cat_cmap = ma.indices[idx_name]
        cat_indices = get_cat_indices(catgroups, cat_cmap)
        cat_names = category_names(catgroups)
        dim_idx = ma._dim_idx[maps_dim]
        if dim_idx == 1 and len(ma.dims) == 2:
            if len(set([len(c) for c in cat_indices])) == 1:
                new_data = ma.data[:, np.array(cat_indices)].sum(axis=-1)
            else:
                new_data = jnp.array([ma.data[:, c].sum(axis=-1) for c in cat_indices])
        else:
            raise Exception("Only timecubes supported currently")
        out_indices = {
            name: (maps, idx)
            for name, (maps, idx) in ma.indices.items()
            if maps != maps_dim
        }
        out_indices["category"] = ("category", cat_names)
        return ManagedArray(new_data, ["time", "category"], indices=out_indices)

    if dims is not None:
        lma = LA(ma.data, ma.dims)
        lsummed = lma.sum(dims)
        out_indices = {
            name: (maps, idx)
            for name, (maps, idx) in ma.indices.items()
            if maps in lsummed.axes
        }
        return ManagedArray(lsummed.data, lsummed.axes, out_indices)

In [ ]:
age_strat.categories()

In [ ]:
cats = [
    [(loc_strat, ["N", "E"]), (age_strat, ["child"])],
    [(loc_strat, ["S", "W"]), (age_strat, ["adult"])],
]

In [ ]:
category_names(cats)

In [ ]:
age_cats[0]

In [ ]:
masum(infdata, cats=("source", age_cats))

In [ ]:
masum(infdata, dims="compartment")

In [ ]:
LA(infdata.data, infdata.dims).axes

In [ ]:
masum(infdata, dims="compartment")

In [ ]:
linf = LA(infdata.data, infdata.dims)

In [ ]:
linf.sum(linf.to_axis("time"))

In [ ]:
infdata.data[:, np.array(cidx)].shape

In [ ]:
infdata.data[:, np.array(cidx)].sum(axis=-1).shape

In [ ]:
infdata.data

In [ ]:
infdata.query(source=age_strat["child"])

In [ ]:
assert False

In [ ]:
epoch.index_to_dti()

In [ ]:
iflow_data = comp_results["flows"]["infection"]

In [ ]:
def sum_over()

In [ ]:
category_idx_reduction?

In [ ]:
iflow_data.indices

In [ ]:
ma = ManagedArray(
    ctvals,
    dims=["time", "compartment"],
    indices={"time": ("time", dti), "compartment": ("compartment", cmap)},
)

In [ ]:
from utils import Epoch

In [ ]:
adti

In [ ]:
ManagedArray(iflow_data, ["time", "flow"], indices={"time": dti})

In [ ]:
class CompartmentalEpiModel:
    def __init__(self, base_compartments):
        base_strat = Stratification("base", base_compartments)
        self.compartment_map = CompartmentMap.new(base_strat)
        self.flows = {}

    def add_transition_flow(self, name, source, dest, param):
        flow = TransitionFlow(source, dest, param)
        self.flows[name] = flow

    def add_infection_frequency_flow(self, name, source, dest):
        self.add_transition_flow(name, source, dest, "foi")

    def stratify(self, stratification, stratifies, mm=None):
        self.compartment_map.stratify(stratification, stratifies)
        if mm is not None:
            self.mm = mm

mm = jnp.ones((len(age_strat.strata), len(age_strat.strata)))


    def foi_mixing(self, cdatamap, params):
        ipops = query_cat_reduction(inf_age_cats, cdatamap)
        total_pop = cdatamap.data.sum()
        age_foi = mm @ ipops / total_pop * params["contact_rate"]
        return CategoryData(age_strat.categories(), age_foi)
        

In [ ]:
infection = TransitionFlow(disease_state["S"], disease_state["I"], "foi")
recovery = TransitionFlow(disease_state["I"], disease_state["R"], "recovery_rate")

In [ ]:
def foi(cdatamap, params):
    ipop = cdatamap.query([disease_state["I"]]).data.sum()
    total_pop = cdatamap.data.sum()
    return (ipop / total_pop) * params["contact_rate"]

In [ ]:
flows = {"infection": infection, "recovery": recovery}

dyn_params = {"foi": foi}

model = NaiveModel(humans, flows, dyn_params)
run = model.get_runner()

istate = jnp.array([100.0, 5.0, 0.0])
params = {"contact_rate": 0.2, "recovery_rate": 0.01}
t = 200

comp_results = run(istate, params, t)

In [ ]:
severity_strat = humans.stratify(
    Stratification("severity", ["mild", "severe"]), disease_state["I"]
)

In [ ]:
humans

In [ ]:
run = model.get_runner()
istate = jnp.array([100.0, 5.0, 0.0, 0.0])
params = {"contact_rate": 0.2, "recovery_rate": 0.01}
t = 200

comp_results = run(istate, params, t)

comp_labels = [compname(c) for c in model.cmap.compartments]
pd.DataFrame(comp_results, columns=comp_labels).plot()

In [ ]:
age_strat = humans.stratify(
    Stratification("age", ["child", "young_adult", "adult", "older"])
)

In [ ]:
inf_age_cats = [
    [disease_state["I"], age_strat[age_group]] for age_group in age_strat.strata
]

inf_age_cats

In [ ]:
mm = jnp.ones((len(age_strat.strata), len(age_strat.strata)))


def foi_mixing(cdatamap, params):
    ipops = query_cat_reduction(inf_age_cats, cdatamap)
    total_pop = cdatamap.data.sum()
    age_foi = mm @ ipops / total_pop * params["contact_rate"]
    return CategoryData(age_strat.categories(), age_foi)

In [ ]:
istate = humans.zeros(np)
s_idx = humans.query(disease_state["S"]).indices
istate.data[s_idx] = np.array((10.0, 20.0, 50.0, 20.0))
i_idx = humans.query(disease_state["I"]).indices
istate.data[i_idx] = 1.0

In [ ]:
foi_mixing(istate, params)

In [ ]:
flows = {"infection": infection, "recovery": recovery}

dyn_params = {"foi": foi_mixing}

model = NaiveModel(humans, flows, dyn_params)

run = model.get_runner()

params = {"contact_rate": 0.2, "recovery_rate": 0.02}
t = 200

comp_results = run(istate.data, params, t)

comp_labels = [compname(c) for c in model.cmap.compartments]
pd.DataFrame(comp_results, columns=comp_labels).plot()